# S43-a -- fine-tune Qwen3 1.7B on the voice-command training set

Four things you do by hand; everything else runs on its own.

1. **Upload two files here** when this notebook's cells ask for them: nothing --
   the training data comes from Google Drive (next cell tells you the folder),
   and the two things you bring back (`predictions.jsonl` and
   `model-q4_k_m.gguf`) are written to that same Drive folder, not uploaded.
   You DO need `score_port.py` (from `scripts/voice/train/` in the repo) in
   this notebook's own file browser (the folder icon on the left) before
   running the scoring cell -- drag it in, or use the upload button there.
2. **Runtime -> Change runtime type -> T4 GPU**, before running anything.
3. **Runtime -> Run all.** About an hour. The first few cells print progress
   in seconds; training is the long step. If the session disconnects, run it
   again -- everything is written to Drive, not to the Colab session, so
   nothing already finished is lost (training itself restarts from scratch,
   there is no mid-run checkpoint resume in this notebook).
4. **When it finishes**, download `predictions.jsonl` and `model-q4_k_m.gguf`
   from the run folder in Drive (the last cell prints the exact path) and
   bring them, plus the score line the scoring cell printed, back to the
   developer session.

See `scripts/voice/train/README.md` in the repo for the full walkthrough,
including what "good" looks like.


In [ ]:
# ---------------------------------------------------------------------------
# Settings -- every hyperparameter in one place (S43-a brief SS4 step 2).
# ---------------------------------------------------------------------------
import time

BASE_MODEL = "Qwen/Qwen3-1.7B"
SEED = 20260912  # fixed everywhere below -- torch, numpy, random, the trainer

EPOCHS = 3
LR = 2e-4

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

MAX_LEN = 512
BATCH = 8
GRAD_ACCUM = 4  # BATCH * GRAD_ACCUM must be the effective batch the brief asks for (32)
EFFECTIVE_BATCH = BATCH * GRAD_ACCUM
assert EFFECTIVE_BATCH == 32, f"expected an effective batch of 32, got {EFFECTIVE_BATCH}"

RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
DATA_DIR = "/content/drive/MyDrive/scheduler-voice/data"
OUT_DIR = f"/content/drive/MyDrive/scheduler-voice/run-{RUN_STAMP}"

print(f"BASE_MODEL={BASE_MODEL}  SEED={SEED}  EPOCHS={EPOCHS}  LR={LR}")
print(f"LoRA: r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"targets={TARGET_MODULES}")
print(f"BATCH={BATCH} x GRAD_ACCUM={GRAD_ACCUM} = effective {EFFECTIVE_BATCH}, MAX_LEN={MAX_LEN}")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUT_DIR={OUT_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Install -- pinned versions (S43-a brief SS4 step 3). See the report for how
# each version below was chosen: confirmed on PyPI on 12 Sept 2026 unless the
# comment beside it says otherwise.
#
# Plain `subprocess` calls, not `!pip` / `%pip` magics -- this keeps every
# cell here syntactically ordinary Python, which is what lets
# `python -m py_compile` check this notebook's code on a machine with no
# GPU and no `torch` at all (S43-a brief SS2).
# ---------------------------------------------------------------------------
# Colab preinstalls TensorFlow. `transformers` must never import it here --
# `is_tf_available()` (called by things like `transformers.set_seed`) would
# otherwise import TensorFlow and whatever protobuf version it wants, which
# broke `transformers.set_seed` the first time this notebook ran end to
# end. Kept as a guard even now that the requirements file which originally
# caused that conflict is no longer installed below. Repeated at the top of
# the next cell too, so it still holds if that cell is ever run on its own.
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import subprocess
import sys

# Colab preinstalls torchao 0.10.0; peft 0.20's LoRA tuner PROBES for
# torchao the moment `get_peft_model()` runs (peft/tuners/lora/torchao.py's
# `is_torchao_available()`) and refuses anything below 0.16.0 -- even
# though this pipeline never quantises with it. Uninstall it outright,
# before any pip install below; never upgrade it in place, since a newer
# torchao pulls its own torch and risks the same GPU-vs-CPU wheel problem
# the requirements-file fix above already guards against.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)


def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


# llama.cpp has no PyPI package and does not use semantic versioning the
# way the packages below do -- there is no version number to "confirm on
# PyPI". Clone HEAD of the default branch; if `convert_hf_to_gguf.py` ever
# stops recognising Qwen3's architecture, that is a llama.cpp-side
# regression to chase there, not a pin to bump here (brief SS2's honesty
# rule -- said plainly rather than guessing a tag that may not exist).
#
# We do NOT install llama.cpp's own requirements file
# (requirements-convert_hf_to_gguf.txt). It pins `torch==2.11.0` from a CPU
# wheel index and, via its included requirements-convert_legacy_llama.txt,
# `transformers==4.57.6` -- either one breaks a GPU training runtime (the
# second Colab finding was exactly this: `ImportError: cannot import name
# 'GenerationMixin' from 'transformers.generation'`, from the transformers
# downgrade). `convert_hf_to_gguf.py` only actually needs `gguf`,
# `sentencepiece`, `numpy`, `safetensors`, and the torch/transformers this
# cell already installs -- numpy comes from Colab's base image and
# safetensors from transformers' own dependencies, so only the first two
# need installing here, BEFORE our five pins below (same reasoning as
# before: this runs first, our pins below still get the last word).
subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", "/content/llama.cpp"],
    check=True,
)
pip_install("gguf>=0.10", "sentencepiece>=0.1.98,<0.3.0")

# Colab ships its own `torch` build matched to the runtime's CUDA version --
# do not pin or reinstall it here. Installing a mismatched torch wheel is
# the single most common way to break a Colab GPU runtime.
pip_install(
    "transformers==5.17.0",   # confirmed on PyPI 12 Sept 2026; requires torch>=2.5, python>=3.10
    "peft==0.20.0",           # confirmed on PyPI 12 Sept 2026 (released 28 Jul 2026)
    "trl==1.13.0",            # confirmed on PyPI 12 Sept 2026; v1.0 (13 Aug 2026) is the
                               # first stable line -- SFTConfig's `assistant_only_loss` is
                               # this cell's reason for wanting >=1.0, not just a recent patch
    "datasets==5.0.1",        # confirmed on PyPI 12 Sept 2026
    "accelerate==1.15.0",     # confirmed on PyPI 12 Sept 2026
)

import torch

print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached to this runtime. In the Colab menu: "
        "Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Assert our five pins actually won the install order above -- a future
# requirements file (this notebook's own, or one llama.cpp adds later)
# fighting a pin fails HERE, loudly, naming the actual version, not three
# cells later as an unexplained ImportError (brief SS2: "no cell swallows
# an error").
import accelerate
import datasets
import peft
import transformers
import trl

PINNED_VERSIONS = {
    "transformers": (transformers, "5.17.0"),
    "peft": (peft, "0.20.0"),
    "trl": (trl, "1.13.0"),
    "datasets": (datasets, "5.0.1"),
    "accelerate": (accelerate, "1.15.0"),
}
for name, (module, expected) in PINNED_VERSIONS.items():
    actual = module.__version__
    assert actual == expected, (
        f"{name} is {actual}, expected {expected} -- a later install in this cell "
        "(most likely a requirements file with its own pin) downgraded or upgraded it "
        "after our pin ran"
    )
print("pins hold:", ", ".join(f"{n}=={m.__version__}" for n, (m, _) in PINNED_VERSIONS.items()))

# Confirm the torchao uninstall near the top of this cell actually took --
# peft's LoRA tuner probes for torchao lazily, at `get_peft_model()` time,
# so nothing above would have caught it coming back (e.g. pulled in again
# as another package's dependency).
import importlib.util

assert importlib.util.find_spec("torchao") is None, (
    "torchao is still installed. peft's LoRA tuner probes for it at get_peft_model() "
    "time and refuses Colab's version (0.10.0, below peft's 0.16.0 floor); this "
    "pipeline never uses torchao, so re-run the uninstall near the top of this cell "
    "instead of upgrading it."
)


In [ ]:
# ---------------------------------------------------------------------------
# Mount Drive and load data (S43-a brief SS4 step 4).
# ---------------------------------------------------------------------------
# Colab preinstalls TensorFlow; keep it out of this cell too (see the
# install cell's comment -- this repeats the guard so this cell stays safe
# to run on its own).
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import json
import random

import numpy as np
from google.colab import drive


def seed_everything(seed):
    """Seeds python's `random`, `numpy`, and `torch` (CPU and every CUDA
    device) by hand -- NOT `transformers.set_seed`, which calls
    `is_tf_available()` and so imports TensorFlow. No TensorFlow path here,
    on purpose (this is the fix for the first Colab run's
    `ImportError: cannot import name 'runtime_version' from
    'google.protobuf'`, eight frames inside `import tensorflow`)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)

drive.mount("/content/drive")

TRAIN_CHAT_PATH = os.path.join(DATA_DIR, "train.chat.jsonl")
HELDOUT_PATH = os.path.join(DATA_DIR, "heldout.jsonl")
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.json")

for path in (TRAIN_CHAT_PATH, HELDOUT_PATH, MANIFEST_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} is missing. Upload the three files `npm run voice:prepare` wrote "
            f"(train.chat.jsonl, heldout.jsonl, manifest.json) to {DATA_DIR} in Drive, "
            "then re-run this cell."
        )

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_chat_rows = load_jsonl(TRAIN_CHAT_PATH)
heldout_rows = load_jsonl(HELDOUT_PATH)

assert len(train_chat_rows) == manifest["trainRows"], (
    f"train.chat.jsonl has {len(train_chat_rows)} rows, manifest.json says {manifest['trainRows']}"
)
assert len(heldout_rows) == manifest["heldoutRows"], (
    f"heldout.jsonl has {len(heldout_rows)} rows, manifest.json says {manifest['heldoutRows']}"
)

train_sentences = {row["messages"][1]["content"] for row in train_chat_rows}
heldout_sentences = {row["sentence"] for row in heldout_rows}
overlap = train_sentences & heldout_sentences
assert not overlap, (
    f"{len(overlap)} sentence(s) appear in BOTH train and held-out, e.g. {next(iter(overlap))!r} "
    "-- the model would be scored on something it trained on"
)

SYSTEM_PROMPT = manifest["systemPrompt"]  # read from the manifest, never retyped here

print(f"train: {len(train_chat_rows)} chat rows   held-out: {len(heldout_rows)} rows")
print(f"git sha this data was generated from: {manifest['gitSha']}")
print(
    "rule-parser baseline (from manifest.json): "
    f"clean {manifest['ruleParserBaseline']['clean'] * 100:.1f}%  "
    f"perturbed {manifest['ruleParserBaseline']['perturbed'] * 100:.1f}%"
)


In [ ]:
# ---------------------------------------------------------------------------
# Tokenizer and base model (S43-a brief SS4 step 5).
# ---------------------------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"loading {BASE_MODEL} in {DTYPE}")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto")

# Thinking mode is OFF in both training and inference (S43-a brief SS2) --
# `enable_thinking=False` on every `apply_chat_template` call in this
# notebook, this one included.
#
# Assert the chat template's own round trip against the manifest's sample
# BEFORE spending an hour training on top of it (brief SS2: "asserts its own
# serialisation against a sample in the manifest").
sample = manifest["sample"]
sample_row = next(r for r in heldout_rows if r["id"] == sample["id"])
assert sample_row["sentence"] == sample["sentence"], "manifest sample does not match heldout.jsonl"

sample_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": sample_row["sentence"]},
]
prompt_text = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
full_text = prompt_text + sample["canonicalForm"]
round_trip_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
decoded = tokenizer.decode(round_trip_ids)
assert sample["canonicalForm"] in decoded, (
    "the tokenizer did not round-trip the manifest's sample form byte for byte -- "
    "the chat template or a special token changed underneath this notebook"
)
print(f"chat-template round trip OK against manifest sample {sample['id']!r}")


In [ ]:
# ---------------------------------------------------------------------------
# LoRA (S43-a brief SS4 step 6).
# ---------------------------------------------------------------------------
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ---------------------------------------------------------------------------
# Train (S43-a brief SS4 step 7).
#
# Loss on the assistant turn only, via TRL 1.x's `SFTConfig(assistant_only_loss=True)`
# -- this is the "say which API" the brief asks for. TRL patches the chat
# template to add `{% generation %}` markers for known model families
# (Qwen3 included) when this flag is set, so the system and user tokens are
# masked out of the loss without a hand-written response template or a
# `DataCollatorForCompletionOnlyLM` (the older, string-matching approach --
# not used here because it is easy to get subtly wrong across tokenizers,
# per TRL's own docs).
# ---------------------------------------------------------------------------
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_dataset = Dataset.from_list([{"messages": row["messages"]} for row in train_chat_rows])

ADAPTER_DIR = os.path.join(OUT_DIR, "adapter")

sft_config = SFTConfig(
    output_dir=os.path.join(OUT_DIR, "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_length=MAX_LEN,
    bf16=(DTYPE == torch.bfloat16),
    fp16=(DTYPE == torch.float16),
    seed=SEED,
    data_seed=SEED,
    logging_steps=20,
    save_strategy="no",
    report_to=[],
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

train_result = trainer.train()
print(train_result.metrics)

os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"adapter saved to {ADAPTER_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Predict on held-out (S43-a brief SS4 step 8).
#
# Greedy decoding; generation is stopped the moment the text generated SO
# FAR closes a complete top-level JSON object (brief SS2: "the generation is
# stopped at the first closing brace of a complete JSON object") -- a
# `StoppingCriteria` that decodes the tokens generated so far on every step
# and counts braces, skipping over quoted strings so a brace inside a name
# never miscounts.
# ---------------------------------------------------------------------------
from transformers import StoppingCriteria, StoppingCriteriaList


def _scan_braces(text):
    """Returns (started, depth, closed_at) for `text`: whether a top-level
    '{' has been seen, the current brace depth, and the index one past a
    complete top-level object's closing '}' if one was found."""
    depth = 0
    started = False
    in_string = False
    escaped = False
    for i, ch in enumerate(text):
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
            started = True
        elif ch == "}":
            depth -= 1
            if started and depth == 0:
                return started, depth, i + 1
    return started, depth, None


class JsonObjectStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.done = False

    def __call__(self, input_ids, scores, **kwargs):
        if self.done:
            return True
        generated_ids = input_ids[0][self.prompt_len :]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        _, _, closed_at = _scan_braces(text)
        if closed_at is not None:
            self.done = True
            return True
        return False


def extract_first_json_object(text):
    _, _, closed_at = _scan_braces(text)
    if closed_at is None:
        return None
    start = text.find("{")
    return text[start:closed_at] if start != -1 else None


model.eval()

predictions = []
n_failed = 0
for row in heldout_rows:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["sentence"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    stopper = JsonObjectStoppingCriteria(tokenizer, prompt_len)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            stopping_criteria=StoppingCriteriaList([stopper]),
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_text = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
    json_text = extract_first_json_object(generated_text)
    form = None
    if json_text is not None:
        try:
            form = json.loads(json_text)
        except json.JSONDecodeError:
            form = None
    if form is None:
        n_failed += 1
    predictions.append({"id": row["id"], "form": form})

PREDICTIONS_PATH = os.path.join(OUT_DIR, "predictions.jsonl")
with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    for p in predictions:
        f.write(json.dumps(p) + "\n")

print(f"wrote {len(predictions)} predictions to {PREDICTIONS_PATH}")
print(f"{n_failed} row(s) failed to produce a parseable JSON object")


In [ ]:
# ---------------------------------------------------------------------------
# Score (S43-a brief SS4 step 9).
#
# `score_port.py` (from `scripts/voice/train/` in the repo) must already be
# in this notebook's own file browser (the folder icon on the left) --
# step 1 of the title cell above. This table is for your eyes in Colab only;
# the run that counts is `npm run voice:score`-shaped, on the developer's
# machine, against THIS predictions.jsonl (brief SS2).
# ---------------------------------------------------------------------------
import importlib.util

SCORE_PORT_PATH = "/content/score_port.py"
if not os.path.exists(SCORE_PORT_PATH):
    raise FileNotFoundError(
        "score_port.py is missing from /content/. Upload it (from "
        "scripts/voice/train/score_port.py in the repo) using the Files pane "
        "on the left, then re-run this cell."
    )

spec = importlib.util.spec_from_file_location("score_port", SCORE_PORT_PATH)
score_port = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score_port)

result = score_port.score(heldout_rows, score_port.predictions_predict(PREDICTIONS_PATH))
score_port.print_table(result)

clean_rate = score_port.rate(result["clean"])
print(f"clean rate {clean_rate * 100:.1f}%")
print(
    "rule-parser baseline (from manifest.json): "
    f"clean {manifest['ruleParserBaseline']['clean'] * 100:.1f}%  "
    f"perturbed {manifest['ruleParserBaseline']['perturbed'] * 100:.1f}%"
)
print(
    "This is Colab's own read. The number that goes on S43's card is "
    "`node scripts/voice/score.mjs --heldout data/voice/heldout.jsonl "
    "--predictions predictions.jsonl --bar 0.95`, run on the developer's machine."
)


In [ ]:
# ---------------------------------------------------------------------------
# Merge and export (S43-a brief SS4 step 10).
#
# `convert_hf_to_gguf.py` below runs inside this same training environment,
# against transformers 5.17 (not the 4.57.6 its own requirements file
# wants -- see the install cell). If it ever objects to that version, the
# fix is a separate venv for this converter step, never a downgrade here.
# ---------------------------------------------------------------------------
MERGED_DIR = os.path.join(OUT_DIR, "merged")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"merged model saved to {MERGED_DIR}")

GGUF_F16_PATH = os.path.join(OUT_DIR, "model-f16.gguf")
subprocess.run(
    [
        sys.executable,
        "/content/llama.cpp/convert_hf_to_gguf.py",
        MERGED_DIR,
        "--outfile",
        GGUF_F16_PATH,
        "--outtype",
        "f16",
    ],
    check=True,
)

# llama.cpp ships no prebuilt `llama-quantize` via pip -- build it.
subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "/content/llama.cpp"], check=True)
subprocess.run(
    ["cmake", "--build", "/content/llama.cpp/build", "--target", "llama-quantize", "-j", "4"],
    check=True,
)

GGUF_Q4_PATH = os.path.join(OUT_DIR, "model-q4_k_m.gguf")
subprocess.run(
    ["/content/llama.cpp/build/bin/llama-quantize", GGUF_F16_PATH, GGUF_Q4_PATH, "Q4_K_M"],
    check=True,
)

f16_size = os.path.getsize(GGUF_F16_PATH)
q4_size = os.path.getsize(GGUF_Q4_PATH)
print(f"f16:    {GGUF_F16_PATH}  ({f16_size / 1e9:.2f} GB)")
print(f"Q4_K_M: {GGUF_Q4_PATH}  ({q4_size / 1e9:.2f} GB)")


## What to bring back

From `{OUT_DIR}` in Drive (the exact path was printed by the Settings cell
above, with today's timestamp):

- `predictions.jsonl`
- `model-q4_k_m.gguf`

Download both into `data/voice/runs/<timestamp>/` on your machine (that
folder is gitignored), plus the score line the scoring cell printed, and
bring them to the developer session -- see `scripts/voice/train/README.md`
step 5 for the exact scoring command that goes on S43's card.
